In [9]:
import os
import re
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [10]:
def extract_precedents(text):
    """
    Identifies cited cases by searching for the 'v.' pattern.
    """
    # Specifically looking for Party A v. Party B
    pattern = r"([A-Z][\w\s\.\&]+ v\. [A-Z][\w\s\.\&]+)"
    found = re.findall(pattern, text)
    # Filter out very short strings that might be false positives
    return list(set([p.strip() for p in found if len(p) > 10]))

In [11]:
def extract_from_html(html_content):
    """
    Parses professional metadata from SCR HTML.
    """
    data = {
        "case_name": None,
        "coram": None, 
        "decision_date": None, 
        "case_no": None, 
        "disposal_nature": None
    }
    
    
    if not html_content: 
        return data
    
    patterns = {
        "case_name": r"<strong>(.*?)</strong>",
        "coram": r"Coram : (.*?)<br>",
        "decision_date": r"Decision Date :</span><font color='green'> (.*?)</font>",
        "case_no": r"Case No :</span><font color='green'> (.*?)</font>",
        "disposal_nature": r"Disposal Nature :</span><font color='green'> (.*?)</font>"
    }
    
    for key, pattern in patterns.items():
        match = re.search(pattern, html_content)
        if match:
            # Strips residual HTML like <strong> or <sup>
            clean_val = re.sub(r'<.*?>', '', match.group(1)).strip()
            data[key] = clean_val
    return data

In [12]:
# def build_final_parquet(text_folder, metadata_root, output_path ):
#     all_records = []
#     # Identify all text files
#     text_files = [f for f in os.listdir(text_folder) if f.endswith('.txt')]

#     for filename in tqdm(text_files, desc="Building Dataset"):
#         # 1. Load the Full Text
#         file_path = os.path.join(text_folder, filename)
#         with open(file_path, 'r', encoding='utf-8') as f:
#             content = f.read()

#         # 2. Derive IDs (Handle the _EN suffix)
#         base_id = filename.replace('_EN.txt', '').replace('.txt', '')
#         year = base_id.split('_')[0]
        
#         # 3. Path for Metadata
#         json_path = os.path.join(metadata_root, year, "metadata", f"{base_id}.json")

#         # 4. Initialize COMPLETE record structure (Restoring all columns)
#         record = {
#             "file_id": base_id,
#             "year": year,
#             "full_text": content,
#             "precedents": extract_precedents(content),
#             "coram": None,
#             "decision_date": None,
#             "case_no": None,
#             "disposal_nature": None,
#             "neutral_citation": None
#         }

#         # 5. Enrich with Metadata if the JSON exists
#         if os.path.exists(json_path):
#             try:
#                 with open(json_path, 'r', encoding='utf-8') as jf:
#                     meta_json = json.load(jf)
#                     # This updates coram, decision_date, case_no, and disposal_nature
#                     record.update(extract_from_html(meta_json.get("raw_html", "")))
#                     record["neutral_citation"] = meta_json.get("nc_display")
#             except Exception as e:
#                 print(f"Warning: Could not parse JSON for {base_id}: {e}")

#         all_records.append(record)

#     # 6. Final Export
#     df = pd.DataFrame(all_records)
#     # Ensure all required columns exist in the DataFrame
#     df.to_parquet(output_path, engine='pyarrow', index=False, compression='snappy')
#     print(f"\nSaved {len(df)} cases to {output_path}")

In [13]:
def build_final_parquet(text_folder, metadata_root, output_path):
    all_records = []
    text_files = [f for f in os.listdir(text_folder) if f.endswith('.txt')]

    for filename in tqdm(text_files, desc="Building Dataset"):
        # 1. Load Full Text
        with open(os.path.join(text_folder, filename), 'r', encoding='utf-8') as f:
            content = f.read()

        # 2. Match IDs (Strip _EN)
        base_id = filename.replace('_EN.txt', '').replace('.txt', '')
        year = base_id.split('_')[0]
        
        # 3. Path for Metadata
        json_path = os.path.join(metadata_root, year, "metadata", f"{base_id}.json")

        # 4. Initialize Metadata first to get Case Name for filtering
        meta_data = {
            "case_name": None, "coram": None, "decision_date": None, 
            "case_no": None, "disposal_nature": None, "neutral_citation": None
        }

        if os.path.exists(json_path):
            try:
                with open(json_path, 'r', encoding='utf-8') as jf:
                    meta_json = json.load(jf)
                    # Extract fields using your updated extract_from_html function
                    meta_data.update(extract_from_html(meta_json.get("raw_html", "")))
                    meta_data["neutral_citation"] = meta_json.get("nc_display")
            except Exception as e:
                print(f"Warning: JSON error for {base_id}: {e}")

        # 5. Extract and Filter Precedents
        # We clean the main case name to ensure better matching
        current_case_clean = meta_data["case_name"].lower() if meta_data["case_name"] else ""
        
        raw_precedents = extract_precedents(content)
        filtered_precedents = []
        
        for p in raw_precedents:
            clean_p = p.replace('\n', ' ').strip()
            # SELF-CITATION FILTER: 
            # If the precedent is essentially the current case name, skip it.
            if current_case_clean and current_case_clean in clean_p.lower():
                continue
            # Also filter out if the precedent is too short to be a real case
            if len(clean_p) > 10:
                filtered_precedents.append(clean_p)

        # 6. Final Record Creation
        record = {
            "file_id": base_id,
            "year": year,
            "case_name": meta_data["case_name"],
            "full_text": content,
            "precedents": filtered_precedents,
            "coram": meta_data["coram"],
            "decision_date": meta_data["decision_date"],
            "case_no": meta_data["case_no"],
            "disposal_nature": meta_data["disposal_nature"],
            "neutral_citation": meta_data["neutral_citation"]
        }
        all_records.append(record)

    # 7. Save for Kaggle
    df = pd.DataFrame(all_records)
    df.to_parquet(output_path, engine='pyarrow', index=False, compression='snappy')
    print(f"\nSaved {len(df)} cases to {output_path}")

In [14]:
start_year = str(input("Enter start year: "))
end_year = str(input("Enter end year: "))

if __name__ == "__main__":
    TEXT_DIR = f"../../My_Datasets/Text_Datasets/texts_{start_year}-{end_year}"
    META_ROOT = f"../../My_Datasets/SC_{start_year}-{end_year}"
    OUTPUT = f"../../My_Datasets/Parquet_Datasets/SC_Parquet_Dataset_{start_year}-{end_year}.parquet"

    build_final_parquet(TEXT_DIR, META_ROOT, OUTPUT)

Building Dataset: 100%|██████████| 4368/4368 [00:10<00:00, 413.22it/s]



Saved 4368 cases to ../../My_Datasets/Parquet_Datasets/SC_Parquet_Dataset_2020-2025.parquet


In [15]:
import pandas as pd

# Load the parquet file
df = pd.read_parquet(f"../../My_Datasets/Parquet_Datasets/SC_Parquet_Dataset_{start_year}-{end_year}.parquet")

# View the first 5 rows
print(df.head())

             file_id  year                                          case_name  \
0  2020_10_1043_1074  2020  M/S. L. R. BROTHERS INDO FLORA LTD. versus COM...   
1  2020_10_1075_1131  2020  M/S BANDEKAR BROTHERS PVT. LTD. & ANR versus P...   
2  2020_10_1132_1150  2020  THE NEW INDIA ASSURANCE COMPANY LIMITED versus...   
3    2020_10_135_237  2020         VINEETA SHARMA versus RAKESH SHARMA & ORS.   
4       2020_10_1_26  2020          BCH ELECTRIC LIMITED versus PRADEEP MEHRA   

                                           full_text  \
0  M/S. L. R. BROTHERS INDO FLORA LTD.\nv.\nCOMMI...   
1  M/S BANDEKAR BROTHERS PVT. LTD. & ANR\nv.\nPRA...   
2  THE NEW INDIA ASSURANCE COMPANY LIMITED\nv.\nS...   
3  VINEETA SHARMA\nv.\nRAKESH SHARMA & ORS.\n(Civ...   
4  BCH ELECTRIC LIMITED\nv.\nPRADEEP MEHRA\n(Civi...   

                                          precedents  \
0  [Uniworth Textiles Limited v. Commissioner of ...   
1  [Surjit Singh v. Balbir Singh, See also Balasu...   
2  [Sang

In [16]:
print(df.iloc[0].T)

file_id                                             2020_10_1043_1074
year                                                             2020
case_name           M/S. L. R. BROTHERS INDO FLORA LTD. versus COM...
full_text           M/S. L. R. BROTHERS INDO FLORA LTD.\nv.\nCOMMI...
precedents          [Uniworth Textiles Limited v. Commissioner of ...
coram                             A.M. KHANWILKAR*, DINESH MAHESHWARI
decision_date                                              01-09-2020
case_no                                    CIVIL APPEAL No. 7157/2008
disposal_nature                                             Dismissed
neutral_citation                                          2020INSC525
Name: 0, dtype: object


In [19]:
caseName = df.loc[0, 'case_name']
print(caseName)

M/S. L. R. BROTHERS INDO FLORA LTD. versus COMMISSIONER OF CENTRAL EXCISE


In [21]:
value = df.loc[0, 'precedents']
print(value)

['Uniworth Textiles Limited v. Commissioner of Central Excise'
 'Commissioner of Central Excise and Customs v. Suresh Synthetics 2007'
 'Union of India & Anr. v. IndusInd Bank Limited & Anr.'
 'S. L. R. BROTHERS INDO FLORA LTD. v. COMMISSIONER OF CENTRAL EXCISE show cause notice under the provisions of the 1962 Act.'
 'S. L. R. BROTHERS INDO FLORA LTD. v. COMMISSIONER OF CENTRAL EXCISE'
 'Vikram Ispat v. Commissioner of Central Excise'
 'Zile Singh v. State of Haryana & Ors.'
 'S. L. R. BROTHERS INDO FLORA LTD. v. COMMISSIONER OF CENTRAL EXCISE Commissioner of Income Tax'
 'Cosco Blossoms Pvt. Ltd v. Commissioner of Customs'
 'State. A Constitution Bench of this Court in Hansraj Gordhandas v. CCE and Customs held that'
 'New Delhi v. Hari Chand Shri Gopal & Ors.'
 'S. L. R. BROTHERS INDO FLORA LTD. v. COMMISSIONER OF CENTRAL EXCISE 3. The EXIM Policy 1997']
